# 01 — Data Audit

Verify data quality before any signal work.

**Run after**: `scripts/fetch_ohlc.py` has populated `data/ohlc_cache/`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

from config import NIFTY50_SYMBOLS, LOOKBACK_SESSIONS
from utils import load_all_ohlc, compute_overnight_returns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## Cell 1 — Load OHLC & date coverage

In [ ]:
ohlc_dict = load_all_ohlc(NIFTY50_SYMBOLS)

coverage = []
for sym, df in ohlc_dict.items():
    n_missing = df['close'].isna().sum()
    coverage.append({
        'symbol': sym,
        'from': df.index.min().date(),
        'to': df.index.max().date(),
        'sessions': len(df),
        'missing_close': n_missing,
    })

cov_df = pd.DataFrame(coverage).sort_values('sessions', ascending=False)
flagged = cov_df[cov_df['missing_close'] > 5]

print(f'Loaded {len(ohlc_dict)} symbols')
print(f'Common date range: {cov_df["from"].min()} → {cov_df["to"].max()}')
print(f'\nSymbols with >5 missing sessions: {len(flagged)}')
display(cov_df)

## Cell 2 — Overnight return distribution

In [ ]:
overnight_df = compute_overnight_returns(ohlc_dict)

all_rets = overnight_df.values.flatten()
all_rets = all_rets[~np.isnan(all_rets)]

# Clip extreme outliers for display (>10% = likely corp action)
clipped = all_rets[(all_rets > -0.10) & (all_rets < 0.10)]

fig, ax = plt.subplots()
ax.hist(clipped, bins=100, edgecolor='none', alpha=0.75, color='steelblue')
ax.axvline(0, color='red', linewidth=1.2, linestyle='--', label='zero')
ax.axvline(clipped.mean(), color='green', linewidth=1.2, linestyle='-', label=f'mean={clipped.mean():.4%}')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.set_title('Overnight Return Distribution (all stocks, clipped ±10%)')
ax.set_xlabel('Overnight Return')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean overnight return:   {clipped.mean():.4%}')
print(f'Median:                  {np.median(clipped):.4%}')
print(f'Std dev:                 {clipped.std():.4%}')
print(f'Outliers excluded (>10%): {len(all_rets) - len(clipped)}')

## Cell 3 — Mean overnight vs intraday return by stock

In [ ]:
intraday_df = pd.DataFrame({
    sym: df['close'] / df['open'] - 1
    for sym, df in ohlc_dict.items()
})

mean_overnight = overnight_df.mean()
mean_intraday  = intraday_df.mean()

compare = pd.DataFrame({
    'overnight': mean_overnight,
    'intraday':  mean_intraday,
}).sort_values('overnight', ascending=False)

fig, ax = plt.subplots(figsize=(16, 5))
x = range(len(compare))
width = 0.4
ax.bar([i - width/2 for i in x], compare['overnight'] * 100, width=width, label='Overnight', color='steelblue', alpha=0.85)
ax.bar([i + width/2 for i in x], compare['intraday']  * 100, width=width, label='Intraday',  color='salmon',   alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(compare.index, rotation=90, fontsize=8)
ax.set_ylabel('Mean Return (%)')
ax.set_title('Mean Overnight vs Intraday Return per Stock')
ax.legend()
plt.tight_layout()
plt.show()

overnight_wins = (mean_overnight > mean_intraday).sum()
print(f'Stocks where overnight > intraday: {overnight_wins}/{len(compare)}')
print('→ If this is < 30/50, check open price data quality.')

## Cell 4 — Key validation: overnight anomaly check

For the strategy to make sense, mean overnight return must exceed mean intraday return for most stocks.

In [ ]:
pct_overnight_wins = overnight_wins / len(compare)
mean_all_overnight = mean_overnight.mean()
mean_all_intraday  = mean_intraday.mean()

print('=== ANOMALY VALIDATION ===')
print(f'Cross-sectional mean overnight return: {mean_all_overnight:.4%}')
print(f'Cross-sectional mean intraday return:  {mean_all_intraday:.4%}')
print(f'Stocks with overnight > intraday:       {overnight_wins}/{len(compare)} ({pct_overnight_wins:.0%})')
print()
if pct_overnight_wins >= 0.60 and mean_all_overnight > 0:
    print('✓ PASS: Overnight anomaly confirmed in your dataset.')
elif mean_all_overnight > 0:
    print('⚠ PARTIAL: Positive mean overnight return but weaker than expected.')
    print('  Check for corporate action contamination (Cell 5).')
else:
    print('✗ FAIL: Mean overnight return is not positive.')
    print('  Most likely cause: adjusted vs unadjusted open prices from Kite.')
    print('  Verify that open prices match NSE bhavcopy for 3-4 spot checks.')

## Cell 5 — Corporate action contamination check

Large overnight returns (>5%) on ex-dividend or split dates inflate scores artificially.

In [ ]:
LARGE_MOVE_THRESHOLD = 0.05  # 5%

large_moves = []
for sym in overnight_df.columns:
    col = overnight_df[sym].dropna()
    suspect = col[col.abs() > LARGE_MOVE_THRESHOLD]
    for dt, ret in suspect.items():
        large_moves.append({'symbol': sym, 'date': dt.date(), 'overnight_return': ret})

lm_df = pd.DataFrame(large_moves).sort_values('overnight_return', key=abs, ascending=False)

print(f'Sessions with |overnight return| > {LARGE_MOVE_THRESHOLD:.0%}: {len(lm_df)}')
print('These are likely ex-dividend/split dates. Review before finalising signal.')
print()
display(lm_df.head(30))

# Proportion of data affected
total_obs = overnight_df.notna().sum().sum()
print(f'\nAffected obs: {len(lm_df)} / {total_obs} ({len(lm_df)/total_obs:.2%})')
print('To exclude these from scoring: filter in compute_scores() where abs(ret) > 0.05')